# Auditoria reproduzível — treinamento controlado

## tl;dr

Este notebook perfila somente os artefatos locais do benchmark: status individuais, métricas por época, métricas de teste, sumários de treino, telemetria e auditorias de dados. Ele não inicia, interrompe ou altera nenhum treinamento.

## Contexto e método

**Grão:** um `status.json` por combinação dataset × batch. Um run `completed` deve ter 100 linhas em `checkpoints/epoch_metrics.csv`, além de métricas de teste e resumo de telemetria. Um run `running` pode ter uma época em curso, ainda sem uma linha de métricas.

### Premissas-chave

- O estado individual do run é a fonte de progresso; `pipeline-status.json` pode refletir um dry-run anterior.
- F1 macro de teste é comparável somente entre batches do mesmo dataset.
- Este notebook assume execução a partir da raiz do repositório.

## Dados e verificações

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import csv
import json

ROOT = Path('outputs/controlled-augmentation2-mac-m4')
BATCH_ROOT = ROOT / 'batch'

runs = []
for status_path in BATCH_ROOT.glob('batch-*/*/runs/*/status.json'):
    status = json.loads(status_path.read_text(encoding='utf-8'))
    parts = status_path.parts
    batch_index = parts.index('batch')
    run_root = status_path.parent
    metrics_path = run_root / 'checkpoints/epoch_metrics.csv'
    rows = list(csv.DictReader(metrics_path.open())) if metrics_path.exists() else []
    test_path = run_root / 'artifacts' / 'test_metrics.json'
    test = json.loads(test_path.read_text()) if test_path.exists() else None
    runs.append({
        'dataset': parts[batch_index + 2],
        'batch_size': int(parts[batch_index + 1].split('-')[1]),
        'status': status['status'],
        'epochs_persisted': len(rows),
        'last_val_macro_f1': float(rows[-1]['val_macro_f1']) if rows else None,
        'test_macro_f1': test['classification']['macro_f1'] if test else None,
    })

print('Status dos batches:', Counter(run['status'] for run in runs))
print('Runs finalizados:', [run for run in runs if run['status'] == 'completed'])
print('Run ativo:', [run for run in runs if run['status'] == 'running'])

In [ ]:
audit_rows = []
for audit_path in sorted((ROOT / 'audit').glob('*/audit/audit.json')):
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    duplicates = audit.get('duplicates', {})
    audit_rows.append({
        'dataset': audit['dataset'],
        'samples': audit['total_samples'],
        'missing_files': len(audit.get('missing_files', [])),
        'unreadable_images': len(audit.get('unreadable_images', [])),
        'invalid_shapes': len(audit.get('invalid_image_shapes', [])),
        'invalid_labels': len(audit.get('invalid_labels', [])),
        'duplicate_excess': duplicates.get('excess_sample_count', 0),
        'cross_label_duplicate_groups': sum(bool(group.get('cross_label')) for group in duplicates.get('groups', [])),
    })

print('Amostras auditadas:', sum(row['samples'] for row in audit_rows))
print('Falhas estruturais:', {key: sum(row[key] for row in audit_rows) for key in ['missing_files', 'unreadable_images', 'invalid_shapes', 'invalid_labels']})
print('Duplicatas:', [row for row in audit_rows if row['duplicate_excess']])

## Resultados

Os resultados só devem ser interpretados para runs `completed`, pois os demais ainda não possuem teste final. A telemetria de cada run concluído deve encerrar com pressão térmica nominal e possuir um `summary.json`; o run ativo deve ter `samples.csv` com atualização recente.

In [ ]:
quality_checks = []
for status_path in BATCH_ROOT.glob('batch-*/*/runs/*/status.json'):
    status = json.loads(status_path.read_text())
    run_root = status_path.parent
    if status['status'] != 'completed':
        continue
    epochs = list(csv.DictReader((run_root / 'checkpoints/epoch_metrics.csv').open()))
    telemetry = json.loads((run_root / 'telemetry' / 'summary.json').read_text())
    quality_checks.append({
        'run': status['run_id'],
        'epochs': len(epochs),
        'test_metrics_exists': (run_root / 'artifacts' / 'test_metrics.json').exists(),
        'thermal_pressure': telemetry['categorical']['thermal_pressure']['current'],
        'telemetry_samples': telemetry['metrics']['cpu_percent']['count'],
    })

assert all(row['epochs'] == 100 for row in quality_checks)
assert all(row['test_metrics_exists'] for row in quality_checks)
assert all(row['thermal_pressure'] == 'nominal' for row in quality_checks)
print('Verificações de completude e telemetria aprovadas para', len(quality_checks), 'runs.')
print('Amostras de telemetria salvas:', sum(row['telemetry_samples'] for row in quality_checks))

## Conclusões

Use os status individuais, os checkpoints e a telemetria para acompanhar o run ativo. Antes de uma comparação final entre todos os datasets, é necessário concluir os batches pendentes e as fases que fizerem parte do comando em execução. Duplicatas com rótulos conflitantes devem permanecer documentadas na análise de CIFAR-100 coarse, FER2013 e SVHN.